# script 2: capa silver con iceberg y messi (parquet-to-iceberg)
aqui leemos el parquet crudo que dejamos en el bucket de taxis del minion con pyarrow. despues nos conectamos al catalogo de messi y creamos la tabla formal de apache iceberg en el bucket `warehouse` (capa silver).


In [ ]:
import os
import s3fs
import pyarrow.parquet as pq
from pyiceberg.catalog import load_catalog

# 1. configuramos s3fs para poder hablar con el minion
s3 = s3fs.S3FileSystem(
    key="admin",
    secret="password",
    client_kwargs={"endpoint_url": "http://minio:9000"}
)

# buscamos el archivo parquet que dejo dlt en el bucket taxis
archivos = s3.glob("taxis/taxis_parquet/df_data/*.parquet")
print(f"archivos parquet encontrados en el minion: {archivos}")
ruta_parquet = f"s3://{archivos[0]}"

# 2. leemos el parquet en memoria con pyarrow de forma columnar
print(f"leyendo datos con pyarrow desde {ruta_parquet}")
tabla_arrow = pq.read_table(ruta_parquet, filesystem=s3)
print(f"total filas leidas: {tabla_arrow.num_rows:,} | columnas: {tabla_arrow.num_columns}")

In [ ]:
# 3. nos conectamos al catalogo de messi
catalog = load_catalog(
    "nessie",
    **{
        "uri": "http://nessie:19120/iceberg/main/",
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
    }
)

# 4. creamos el namespace demo si no existe todavia
namespaces = [ns[0] for ns in catalog.list_namespaces()]
if "demo" not in namespaces:
    catalog.create_namespace("demo")
    print("namespace demo creado en messi")
else:
    print("el namespace demo ya existia en messi")

print("namespaces en el catalogo:", catalog.list_namespaces())

In [ ]:
# 5. creamos y cargamos la tabla de apache iceberg
table_identifier = "demo.taxis_iceberg"

try:
    iceberg_table = catalog.load_table(table_identifier)
    print(f"la tabla {table_identifier} ya existia, le hacemos append")
except Exception:
    print(f"creando tabla nueva {table_identifier} en iceberg...")
    iceberg_table = catalog.create_table(
        table_identifier,
        schema=tabla_arrow.schema
    )

# escribimos los datos en formato iceberg
iceberg_table.append(tabla_arrow)
print("datos cargados a la tabla iceberg melo")

# verificamos cuantas filas quedaron registradas
conteo_iceberg = iceberg_table.scan().to_arrow().num_rows
print(f"total filas en la tabla iceberg ({table_identifier}): {conteo_iceberg:,}")